In [1]:
import pandas as pd
from statsmodels.stats.multitest import multipletests

df = pd.read_excel("McNemar_Test_Results.xlsx")

PROMPTS = ["Zero-shot", "CoT", "Structured"]


def split_configuration(name):
    """'ChatGPT Zero-shot' -> ('ChatGPT', 'Zero-shot')."""
    name = name.strip()
    for prompt in PROMPTS:
        if name.endswith(prompt):
            return name[: -len(prompt)].strip(), prompt
    raise ValueError(f"cannot parse configuration: {name!r}")


rows = []
for _, r in df.iterrows():
    left, right = r["Comparison"].split(" vs ")
    model_1, prompt_1 = split_configuration(left)
    model_2, prompt_2 = split_configuration(right)
    if prompt_1 == prompt_2 and model_1 != model_2:
        family = "model (prompt held constant)"
    elif model_1 == model_2 and prompt_1 != prompt_2:
        family = "prompt (model held constant)"
    else:
        family = "confounded (both differ)"
    rows.append({"Comparison": r["Comparison"], "Family": family, "p_raw": r["p-value"]})

classified = pd.DataFrame(rows)
planned = classified[classified["Family"] != "confounded (both differ)"].copy()

print(f"{len(classified)} comparisons computed")
print(f"{len(planned)} pre-planned and corrected over")
print(f"{len(classified) - len(planned)} confounded, reported in Appendix A, not corrected over")
assert len(classified) == 36, "expected all 36 pairwise comparisons"
assert len(planned) == 18, "expected exactly 18 pre-planned comparisons"

reject, p_holm, _, _ = multipletests(planned["p_raw"], alpha=0.05, method="holm")
planned["p_holm"] = p_holm
planned["significant_raw"] = planned["p_raw"] < 0.05
planned["significant_holm"] = reject

planned = planned.sort_values("p_raw").reset_index(drop=True)
print()
print(planned.to_string(index=False))

n_raw = int(planned["significant_raw"].sum())
n_holm = int(planned["significant_holm"].sum())
changed = planned[planned["significant_raw"] != planned["significant_holm"]]

print()
print(f"significant at 0.05 before Holm: {n_raw}")
print(f"significant at 0.05 after  Holm: {n_holm}")
print(f"comparisons whose status changed: {len(changed)}")
if len(changed):
    print(changed.to_string(index=False))
else:
    print("No result changes significance status under the correction, as reported in Section VII-E.")

planned.to_excel("McNemar_Holm_Adjusted.xlsx", index=False)
print("\nWrote McNemar_Holm_Adjusted.xlsx")

36 comparisons computed
18 pre-planned and corrected over
18 confounded, reported in Appendix A, not corrected over

                             Comparison                       Family        p_raw       p_holm  significant_raw  significant_holm
               ChatGPT CoT vs Llama CoT model (prompt held constant) 3.098473e-12 5.577251e-11             True              True
   ChatGPT Zero-shot vs Llama Zero-shot model (prompt held constant) 3.785517e-11 6.435380e-10             True              True
  ChatGPT Zero-shot vs Gemini Zero-shot model (prompt held constant) 9.290972e-11 1.486556e-09             True              True
ChatGPT Structured vs Gemini Structured model (prompt held constant) 9.925612e-09 1.488842e-07             True              True
              ChatGPT CoT vs Gemini CoT model (prompt held constant) 3.184149e-08 4.457808e-07             True              True
 ChatGPT Structured vs Llama Structured model (prompt held constant) 6.669779e-07 8.670712e-06         